In [10]:
from herbie import Herbie

# Inicializar Herbie para una ventana temporal de prueba (Resolución de 0.25 grados)
H = Herbie(
    "2026-05-10 00:00",
    model="gfs",
    product="pgrb2.0p25",
    fxx=0
)

# Mostrar metadatos del objeto Herbie
H

✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-May-10 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ local


▌▌Herbie GFS model pgrb2.0p25 product initialized 2026-May-10 00:00 UTC F00 ┊ source=aws

In [11]:
# Descargar y estructurar el inventario de variables del GFS
inv = H.inventory()

# Inspeccionar las primeras filas del inventario
inv.head()

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this
0,1,0,884863.0,0-884863,2026-05-10,2026-05-10,PRMSL,mean sea level,anl,:PRMSL:mean sea level:anl
1,2,884864,977942.0,884864-977942,2026-05-10,2026-05-10,CLMR,1 hybrid level,anl,:CLMR:1 hybrid level:anl
2,3,977943,1204333.0,977943-1204333,2026-05-10,2026-05-10,ICMR,1 hybrid level,anl,:ICMR:1 hybrid level:anl
3,4,1204334,1475420.0,1204334-1475420,2026-05-10,2026-05-10,RWMR,1 hybrid level,anl,:RWMR:1 hybrid level:anl
4,5,1475421,1570591.0,1475421-1570591,2026-05-10,2026-05-10,SNMR,1 hybrid level,anl,:SNMR:1 hybrid level:anl


In [12]:
# 1. Definir variables atmosféricas clave para el modelo predictivo de nubosidad
variables_interes = [
    "RH",    # Relative Humidity (%)
    "TMP",   # Temperature (K)
    "UGRD",  # U-Component of Wind (m/s)
    "VGRD",  # V-Component of Wind (m/s)
    "VVEL",  # Vertical Velocity / Omega (Pa/s)
    "PWAT",  # Precipitable Water (kg/m²)
    "TCDC",  # Total Cloud Cover (%)
    "HCDC"   # High Cloud Cover (%)
]

# 2. Control de calidad del inventario GFS
total_vars = inv["variable"].nunique()
vars_encontradas = [v for v in variables_interes if v in inv["variable"].values]

print(f"El inventario completo contiene {total_vars} variables únicas de GFS.")
print(f"Variables de interés localizadas: {len(vars_encontradas)} de {len(variables_interes)}")

El inventario completo contiene 69 variables únicas de GFS.
Variables de interés localizadas: 8 de 8


In [13]:
# Filtrar el inventario por nuestra selección
df_interes = inv[inv["variable"].isin(variables_interes)]

print("======================================================================")
print("             ESTRUCTURA DE NIVELES POR VARIABLE SELECCIONADA          ")
print("======================================================================")

# Agrupar por variable y listar sus niveles de forma compacta
for var, group in df_interes.groupby("variable"):
    niveles_unicos = list(group["level"].unique())
    n_niveles = len(niveles_unicos)
    
    print(f"\n Variable: {var:5s} | Niveles detectados: {n_niveles}")
    
    # Si la variable tiene un perfil vertical denso (como niveles de presión), truncamos la vista
    if n_niveles > 8:
        print(f" └─ Niveles (Muestra): {niveles_unicos[:6]} ... e indicando {n_niveles - 6} niveles más.")
    else:
        print(f" └─ Niveles: {niveles_unicos}")

             ESTRUCTURA DE NIVELES POR VARIABLE SELECCIONADA          

 Variable: HCDC  | Niveles detectados: 1
 └─ Niveles: ['high cloud layer']

 Variable: PWAT  | Niveles detectados: 1
 └─ Niveles: ['entire atmosphere (considered as a single layer)']

 Variable: RH    | Niveles detectados: 51
 └─ Niveles (Muestra): ['0.01 mb', '0.02 mb', '0.04 mb', '0.07 mb', '0.1 mb', '0.2 mb'] ... e indicando 45 niveles más.

 Variable: TCDC  | Niveles detectados: 23
 └─ Niveles (Muestra): ['50 mb', '100 mb', '150 mb', '200 mb', '250 mb', '300 mb'] ... e indicando 17 niveles más.

 Variable: TMP   | Niveles detectados: 54
 └─ Niveles (Muestra): ['0.01 mb', '0.02 mb', '0.04 mb', '0.07 mb', '0.1 mb', '0.2 mb'] ... e indicando 48 niveles más.

 Variable: UGRD  | Niveles detectados: 58
 └─ Niveles (Muestra): ['planetary boundary layer', '0.01 mb', '0.02 mb', '0.04 mb', '0.07 mb', '0.1 mb'] ... e indicando 52 niveles más.

 Variable: VGRD  | Niveles detectados: 58
 └─ Niveles (Muestra): ['planetary bo

In [ ]:
inv[inv["variable"] == "PWAT"][["variable", "level", "search_this"]]